# Fase 1

## Comprender cada tabla y campo

In [ ]:
import pandas as pd, glob
import utils_fase2 as uf2
import os

In [ ]:
def perfil(nombre_tabla, ruta_glob):
    print(f"\n{'='*50}")
    print(f"PERFILANDO TABLA: {nombre_tabla.upper()}")
    print(f"{'='*50}")
    
    archivos = glob.glob(ruta_glob, recursive=True)
    
    if not archivos:
        print(f" No se encontraron archivos para la ruta: {ruta_glob}")
        return None
        
    print(f"Archivos encontrados: {len(archivos)}")
    
    # Validar que todas las particiones tengan las mismas columnas
    cols = {f: tuple(pd.read_csv(f, nrows=0).columns) for f in archivos}
    print("Esquemas distintos entre archivos:", len(set(cols.values())))
    
    # Concatenar todos los archivos encontrados
    df = pd.concat((pd.read_csv(f, low_memory=False) for f in archivos), ignore_index=True)
    
    print("Filas totales:", len(df))
    
    # Generar resumen
    resumen = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "%nulos": (df.isna().mean() * 100).round(1),
        "distintos": df.nunique(),
        "ejemplo": df.iloc[0] if len(df) > 0 else None,
    })
    
    print("\n--- Resumen de Columnas ---")
    print(resumen.to_string())
    
    return df

# ==========================================
# EJECUCIÓN PARA CADA TABLA
# ==========================================

# 1. Tablas Satélite / Maestras (Archivos únicos)
# Nota: Si el archivo está en la raíz usa "customers.csv", si está en carpeta usa "customers/customers.csv"
df_customers = perfil("Customers", "customers.csv")
df_products  = perfil("Products", "products/products.csv")
df_branches  = perfil("Branches", "branches/branches.csv") 

# 2. Tablas Particionadas (Múltiples archivos en subcarpetas)
# El patrón **/*.csv penetra toda la estructura year=X/month=Y/day=Z
df_complaints   = perfil("Complaints", "complaints/**/*.csv")
df_transcripts  = perfil("Call Transcripts", "call_transcripts/**/*.csv")
# La tabla transaccional tiene ~5 millones de registros. 
# df_call_center_interactions ~ 800,000 registros
df_transactions = perfil("Transactions", "transactions/**/*.csv")
df_call_center_interactions = perfil("Call Center Interactions", "call_center_interactions/**/*.csv")

In [ ]:
def valores(df, nombre, max_card=30):
    print(f"\n{'='*50}")
    print(f"=== REVISIÓN PROFUNDA: {nombre.upper()} ===")
    print(f"{'='*50}")
    
    # Valores de columnas categóricas / de baja cardinalidad
    for c in df.columns:
        if df[c].nunique() <= max_card:
            print(f"\n{c}:\n{df[c].value_counts(dropna=False).to_string()}")
            
    # Estadísticas numéricas
    print("\n--- Estadísticas Numéricas ---")
    try:
        print(df.describe().T[["min", "max", "mean"]])
    except KeyError:
        print("No hay columnas numéricas para describir.")
        
    # Rangos de fechas
    print("\n--- Rangos de Fechas ---")
    for c in [c for c in df.columns if "date" in c.lower()]:
        f = pd.to_datetime(df[c], errors="coerce")
        print(f"{c}: min = {f.min()}, max = {f.max()}")

# ==========================================
# 1. REVISIÓN GENERAL POR TABLA
# ==========================================
valores(df_customers, "Customers")
valores(df_branches, "Branches")
valores(df_products, "Products")
valores(df_complaints, "Complaints")
valores(df_transcripts, "Call Transcripts")
valores(df_transactions, "Transactions")
valores(df_call_center_interactions, "Call Center Interactions")

# ==========================================
# 2. CRUCES ESPECÍFICOS Y COMPROBACIÓN DE HIPÓTESIS
# ==========================================
print(f"\n{'='*50}")
print("CRUCES ESPECÍFICOS Y COMPROBACIÓN DE HIPÓTESIS")
print(f"{'='*50}")

print("\n1. Transactions: ¿Nulos en amount_usd correlacionan con la moneda?")
print(pd.crosstab(df_transactions['currency'], df_transactions['amount_usd'].isna(), rownames=['currency'], colnames=['amount_usd_is_NaN']))

print("\n2. Transactions: ¿Nulos en branch_id correlacionan con el canal?")
print(pd.crosstab(df_transactions['channel'], df_transactions['branch_id'].isna(), rownames=['channel'], colnames=['branch_id_is_NaN']))

print("\n3. Customers: ¿Nulos en detected_accent se distribuyen igual por país? (Sospecha de 'neutral')")
print(pd.crosstab(df_customers['country'], df_customers['detected_accent'].isna(), rownames=['country'], colnames=['detected_accent_is_NaN']))

print("\n4. Call Transcripts: Frecuencia de los 42 textos únicos (verificando diversidad)")
print(df_transcripts['customer_text'].value_counts(dropna=False).to_string())

print("\n5. Complaints: ¿Nulos en resolution_date dependen del status?")
print(pd.crosstab(df_complaints['status'], df_complaints['resolution_date'].isna(), rownames=['status'], colnames=['resolution_date_is_NaN']))

print("\n6. Call Center Interactions: Distribución de filas por día (buscando llegadas tardías/vacíos)")
print(df_call_center_interactions.groupby("process_date").size().describe())

print("\n7. Call Center Interactions: Proporción donde contact_reason == reason_category")
print((df_call_center_interactions['contact_reason'] == df_call_center_interactions['reason_category']).mean())

## Resumen de hallazgos y decisiones

Los cruces realizados sobre las diferentes tablas permiten confirmar varias hipótesis iniciales y, al mismo tiempo, ajustar la estrategia de Machine Learning e IA para el Hackaton.

### 1. Call Transcripts: limitación para NLP

- Se identificaron únicamente **42 textos únicos** en `Call Transcripts`.
- Los textos presentan poca variabilidad y corresponden principalmente a plantillas de consultas de saldo.
- No se encontraron conversaciones relacionadas con **"cargo no reconocido"**.
- Por lo tanto, este histórico no ofrece suficiente diversidad para entrenar de forma confiable un modelo de clasificación de intenciones o implementar un RAG complejo basado en estos datos.

#### Decisión

El uso de IA generativa debe enfocarse principalmente en el **flujo conversacional en tiempo real**, por ejemplo:

- Entender el mensaje del cliente.
- Aclarar montos, fechas o transacciones ambiguas.
- Extraer información relevante.
- Estructurar la información en un JSON.
- Entregar los datos estructurados al agente o motor de decisión.

### 2. Nulos estructurales y reglas de negocio
`branch_id`

Los nulos de branch_id se concentran principalmente en canales digitales como:

App

Web

POS

Transfer

Esto puede corresponder a un comportamiento esperado del negocio: una interacción digital no necesariamente tiene una sucursal física asociada.

`detected_accent`

Aproximadamente el 30 % de los registros presenta detected_accent nulo, con una distribución similar entre México, Colombia y Argentina.

De acuerdo con el diccionario de datos, estos valores representan la categoría Neutral.

#### Decisión
No se deben tratar estos casos automáticamente como errores de calidad. Mapear a -> **Neutral**

### 3. Limpieza estructural
`contact_reason`

El análisis muestra que contact_reason es un duplicado exacto de `reason_category`, con una proporción de igualdad de 1.0.

#### Decisión
Eliminar una de las dos columnas, siempre que no exista una dependencia funcional o contractual que obligue a conservarla.

Esto reduce redundancia y simplifica el modelo de datos.

**Fechas de resolución**

Se identificaron:

140 casos Closed sin `resolution_date`.

632 casos Resolved sin `resolution_date`.

Estos registros presentan una inconsistencia entre el estado del caso y la fecha de resolución.

Decisión
Antes de eliminar registros, validar si `last_updated` puede utilizarse como fuente confiable para reconstruir la fecha.
### 4. Inconsistencias
`registration_branch_id` en Customers presenta aproximadamente 150,000 valores distintos. Realmente contiene aproximadamente 350 sucursales físicas.

Validar su relación contra el catálogo maestro de sucursales.

### 5. Llegadas tardías de datos
**Implicación**:
Los pipelines de datos y las métricas deben considerar el concepto de late-arriving data para evitar interpretar una partición incompleta como una caída real en el volumen de interacciones.



### 6. Discrepancias entre el diccionario y los datos

| Aspecto | Diccionario | Datos reales |
| :--- | :--- | :--- |
| **Filas de hechos** | complaints 80k, transcripts 200k, transactions 5M, interactions 800k | 67k (84%), 171k (86%), 4.43M (88%), 686k (86%) |
| **Dimensiones** | Coinciden exactamente | `customers`, `products` y `branches` coinciden |
| **document_type** | 5 valores en inglés | 4 valores en español (Pasaporte) |
| **detected_accent** | 4 valores (incluye neutral) | 3 valores más ~30% nulos |
| **currency** | 4 (con USD) | 3 en `products` |
| **transaction_country** | Países del banco | 7 valores (los clientes solo tienen 3 países) |
| **detected_sentiment** | 4 valores | 5 valores |
| **duration_seconds** *(en transcripts)*| NOT NULL | 14% nulo |

# Fase 2 y Fase 3

## Comprender relaciones entre tablas y la calidad de los datos

In [ ]:
def analizar_llaves_huerfanas(df_cust, df_branch):
    print(f"{'='*50}")
    print("ANÁLISIS DE LLAVES FORÁNEAS: REGISTRATION_BRANCH_ID")
    print(f"{'='*50}")
    
    # Extraer valores únicos ignorando nulos (si los hubiera)
    llaves_clientes = set(df_cust['registration_branch_id'].dropna())
    llaves_maestras = set(df_branch['branch_id'].dropna())
    
    # Operaciones de conjuntos
    coincidencias = llaves_clientes.intersection(llaves_maestras)
    huerfanos = llaves_clientes.difference(llaves_maestras)
    
    # Resumen cuantitativo
    print(f"Valores únicos en Customers (registration_branch_id): {len(llaves_clientes)}")
    print(f"Valores únicos en Branches (branch_id):               {len(llaves_maestras)}")
    print(f"Llaves que hacen MATCH:                               {len(coincidencias)}")
    print(f"Llaves HUÉRFANAS (Ruptura referencial):               {len(huerfanos)}")
    
    # Validación de formato (muestreo)
    print("\n--- Patrones de Formato ---")
    print("Ejemplo de llaves VÁLIDAS en Branches:")
    print(list(llaves_maestras)[:5])
    
    if huerfanos:
        print("\nEjemplo de llaves HUÉRFANAS (solo en Customers):")
        print(list(huerfanos)[:5])
        
    # Validar qué pasa con los productos (que también tienen branch_id)
    # Asumiendo que df_products está disponible en tu entorno
    if 'df_products' in globals():
        llaves_productos = set(df_products['opening_branch_id'].dropna())
        huerfanos_prod = llaves_productos.difference(llaves_maestras)
        print(f"\nNota: En Products hay {len(huerfanos_prod)} llaves huérfanas de {len(llaves_productos)} únicas.")

# Ejecutar la función
analizar_llaves_huerfanas(df_customers, df_branches)

### Hallazgo: `registration_branch_id` en `Customers`

El análisis confirma que `registration_branch_id` presenta una **inconsistencia de integridad referencial** y no debe considerarse una llave foránea válida hacia `Branches`.

#### Evidencia

- Se identificaron **149,995 registros huérfanos**.
- Los valores presentan códigos únicos con prefijo `SUC-`, lo que sugiere que fueron generados de forma sintética.
- La relación esperada entre `Customers.registration_branch_id` y `Branches.branch_id` no se cumple.

#### Impacto en el modelado

Utilizar esta columna para relacionar `Customers` con `Branches` puede generar una pérdida masiva de información.

La validación de **Products** muestra que `opening_branch_id` no presenta registros huérfanos respecto a Branches.

Por lo tanto, cuando el caso de uso requiera relacionar un cliente con una sucursal física, la lógica puede utilizar la relación:

**Customers**  
↓  
**Products**  
↓  
`opening_branch_id`  
↓  
**Branches**



 **Análisis transversal del modelo de datos**.


## Configuracion utils_fase2
titulo() → organiza visualmente cada sección del diagnóstico.

_s() → convierte valores a texto y representa explícitamente los nulos como NULL.

ct() → genera tablas de contingencia (crosstab).

sin_acentos() → normaliza texto para comparaciones donde los acentos no deben afectar el resultado.

huerfanos() → identifica registros cuyos identificadores no tienen correspondencia en la tabla padre.

fase2() → ejecuta el diagnóstico completo.

In [ ]:
# Habilitar la recarga automática del módulo
%load_ext autoreload
%autoreload 2

# Ejecutar la función principal de diagnóstico
uf2.fase2(
    cust=df_customers, 
    prod=df_products, 
    br=df_branches, 
    comp=df_complaints, 
    tr=df_transcripts, 
    tx=df_transactions, 
    ci=df_call_center_interactions
)

#### 1. Desfase temporal de las transacciones.
#### 2. Inconsistencia entre el cliente que reclama y el producto afectado.
#### 3. Comportamiento de `fraud_score` y posibles umbrales de decisión.


In [ ]:
uf2.investigar_anomalias_fase3(df_customers, df_products, df_complaints, df_transactions)

# Fase 4: Reglas de Limpieza y Transformación (Decisiones Documentadas)

In [ ]:
# ==============================================================================
# FASE 4: ETL Y LIMPIEZA (CREACIÓN DE DATAFRAMES LIMPIOS)
# ==============================================================================

# 1. Crear copias independientes para no mutar los datos originales
df_branches_clean = df_branches.copy()
df_transactions_clean = df_transactions.copy()
df_call_center_interactions_clean = df_call_center_interactions.copy()
df_complaints_clean = df_complaints.copy()
df_customers_clean = df_customers.copy()
df_transcripts_clean = df_transcripts.copy()
df_products_clean = df_products.copy()

# ==========================================
# 2. ELIMINACIÓN DE COLUMNAS (DROP)
# ==========================================
# Coordenadas inválidas
df_branches_clean.drop(columns=['latitude', 'longitude'], errors='ignore', inplace=True)
df_transactions_clean.drop(columns=['latitude', 'longitude'], errors='ignore', inplace=True)

# Duplicados exactos de información
df_transactions_clean.drop(columns=['merchant_category'], errors='ignore', inplace=True)
df_call_center_interactions_clean.drop(columns=['contact_reason'], errors='ignore', inplace=True)

# Columna 100% nula
df_complaints_clean.drop(columns=['origin_interaction_id'], errors='ignore', inplace=True)


# ==========================================
# 3. IMPUTACIÓN DE VALORES FALTANTES (FILL)
# ==========================================
# Imputar acentos con 'neutral'
if 'detected_accent' in df_customers_clean.columns:
    df_customers_clean['detected_accent'] = df_customers_clean['detected_accent'].fillna('neutral')

if 'detected_accent' in df_transcripts_clean.columns:
    df_transcripts_clean['detected_accent'] = df_transcripts_clean['detected_accent'].fillna('neutral')

if 'customer_detected_accent' in df_call_center_interactions_clean.columns:
    df_call_center_interactions_clean['customer_detected_accent'] = df_call_center_interactions_clean['customer_detected_accent'].fillna('neutral')
    
if 'agent_used_accent' in df_call_center_interactions_clean.columns:
    df_call_center_interactions_clean['agent_used_accent'] = df_call_center_interactions_clean['agent_used_accent'].fillna('neutral')


# ==========================================
# 4. CORRECCIÓN Y NORMALIZACIÓN (CLEAN)
# ==========================================
# Estandarizar país en transacciones
mapeo_paises = {'Mexico': 'México', 'MEXICO': 'México', 'mexico': 'México'}
df_transactions_clean['transaction_country'] = df_transactions_clean['transaction_country'].replace(mapeo_paises)

# Deduplicación de product_number en productos
df_products_clean['product_number'] = df_products_clean['product_number'].astype(str)
mascara_duplicados = df_products_clean.duplicated(subset=['product_number'], keep='first')

if mascara_duplicados.any():
    # Romper la colisión agregando el índice al final del string
    sufijos = "-DUP" + df_products_clean.loc[mascara_duplicados].index.astype(str)
    df_products_clean.loc[mascara_duplicados, 'product_number'] = df_products_clean.loc[mascara_duplicados, 'product_number'] + sufijos


# ==========================================
# 5. MARCADOS ANALÍTICOS (FLAGS DE DESFASE)
# ==========================================
# Extraer fechas base de los dataframes limpios
reg = df_customers_clean.drop_duplicates("customer_id").set_index("customer_id")["registration_date"]
ape = df_products_clean.drop_duplicates("product_id").set_index("product_id")["opening_date"]

# Crear boolean flags para transacciones
df_transactions_clean['flag_tx_antes_registro'] = df_transactions_clean['transaction_date'] < df_transactions_clean['customer_id'].map(reg)
df_transactions_clean['flag_tx_antes_apertura'] = df_transactions_clean['transaction_date'] < df_transactions_clean['product_id'].map(ape)


# ==========================================
# 6. RESUMEN DE VERIFICACIÓN
# ==========================================
print("Copias limpias generadas con éxito.")
print(f"-> Dimensiones df_transactions original: {df_transactions.shape}")
print(f"-> Dimensiones df_transactions_clean:  {df_transactions_clean.shape} (menos columnas inútiles, más boolean flags)")
print(f"-> Países en df_transactions original: {df_transactions['transaction_country'].nunique()}")
print(f"-> Países en df_transactions_clean:    {df_transactions_clean['transaction_country'].nunique()} (Normalizados)")

In [ ]:
# ==============================================================================
# VERIFICACIÓN DEL PIPELINE DE LIMPIEZA
# ==============================================================================
print(f"{'='*70}\n1. VERIFICACIÓN DE ELIMINACIÓN (DROP)\n{'-'*70}")
print(f"Branches lat/lon original:   {'latitude' in df_branches.columns}")
print(f"Branches lat/lon limpio:     {'latitude' in df_branches_clean.columns}")
print(f"Tx merchant_category crudo:  {'merchant_category' in df_transactions.columns}")
print(f"Tx merchant_category limpio: {'merchant_category' in df_transactions_clean.columns}")
print(f"Quejas origin_int_id crudo:  {'origin_interaction_id' in df_complaints.columns}")
print(f"Quejas origin_int_id limpio: {'origin_interaction_id' in df_complaints_clean.columns}")

print(f"\n{'='*70}\n2. VERIFICACIÓN DE IMPUTACIÓN (FILL)\n{'-'*70}")
print("Nulos en 'detected_accent' (Customers):")
print(f"   - Original: {df_customers['detected_accent'].isna().sum()} nulos")
print(f"   - Limpio:   {df_customers_clean['detected_accent'].isna().sum()} nulos")
print("Nulos en 'customer_detected_accent' (Interactions):")
print(f"   - Original: {df_call_center_interactions['customer_detected_accent'].isna().sum()} nulos")
print(f"   - Limpio:   {df_call_center_interactions_clean['customer_detected_accent'].isna().sum()} nulos")

print(f"\n{'='*70}\n3. VERIFICACIÓN DE NORMALIZACIÓN Y DEDUPLICACIÓN (CLEAN)\n{'-'*70}")
paises_crudos = sorted(df_transactions['transaction_country'].dropna().unique().tolist())
paises_limpios = sorted(df_transactions_clean['transaction_country'].dropna().unique().tolist())
print(f"Países Originales: {paises_crudos}")
print(f"Países Limpios:    {paises_limpios}")

dups_crudos = df_products.duplicated(subset=['product_number']).sum()
dups_limpios = df_products_clean.duplicated(subset=['product_number']).sum()
print(f"\nDuplicados en product_number original: {dups_crudos}")
print(f"Duplicados en product_number limpio:   {dups_limpios}")

print(f"\n{'='*70}\n4. VERIFICACIÓN DE MARCADOS (FLAGS)\n{'-'*70}")
if 'flag_tx_antes_registro' in df_transactions_clean.columns:
    viajes_registro = df_transactions_clean['flag_tx_antes_registro'].sum()
    viajes_apertura = df_transactions_clean['flag_tx_antes_apertura'].sum()
    print(f"Nuevas columnas booleanas creadas exitosamente.")
    print(f" - Transacciones antes del registro: {viajes_registro:,} ({(viajes_registro/len(df_transactions_clean)):.2%})")
    print(f" - Transacciones antes de apertura:  {viajes_apertura:,} ({(viajes_apertura/len(df_transactions_clean)):.2%})")
else:
    print("Error: No se encontraron las columnas flag_ en df_transactions_clean.")

In [ ]:
cols_fecha = [
    (df_customers_clean, ["registration_date", "last_updated", "date_of_birth"]),
    (df_products_clean, ["opening_date", "expiration_date", "last_updated", "last_transaction_date"]),
    (df_transactions_clean, ["transaction_date", "process_date"]),
    (df_call_center_interactions_clean, ["interaction_date", "process_date"]),
    (df_transcripts_clean, ["process_date"]),
    (df_complaints_clean, ["creation_date", "process_date", "assignment_date",
                           "first_response_date", "resolution_date", "closing_date"]),
    (df_branches_clean, ["branch_opening_date"])
]

for df, cols in cols_fecha:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")


# 5.2. Códigos y Textos leídos como Floats (VARCHAR)
# Corregir response_code (Ej: 14.0 -> "14", NaN -> nulo de pandas)
if 'response_code' in df_transactions_clean.columns:
    df_transactions_clean['response_code'] = (
        df_transactions_clean['response_code']
        .astype("Int64")     # Quita el decimal (este sí es puramente numérico)
        .astype("string")    # Convierte a texto
    )

# Corregir códigos postales (Alfanuméricos en Argentina, ej: 'B1900')
if 'postal_code' in df_customers_clean.columns:
    df_customers_clean['postal_code'] = df_customers_clean['postal_code'].astype("string")
if 'postal_code' in df_branches_clean.columns:
    df_branches_clean['postal_code'] = df_branches_clean['postal_code'].astype("string")


# 5.3. Enteros con valores nulos (INTEGER)
# Pandas usa "Int64" (con mayúscula) para permitir enteros con NaNs.
cols_enteros = [
    (df_call_center_interactions_clean, ["duration_seconds", "wait_time_seconds"]),
    (df_transcripts_clean, ["duration_seconds"]),
    (df_complaints_clean, ["resolution_days", "resolution_satisfaction"])
]

for df, cols in cols_enteros:
    for c in cols:
        if c in df.columns:
            df[c] = df[c].astype("Int64")


# 5.4. Booleanos puros (BOOLEAN)
# Convertir explícitamente a tipo boolean para evitar float64 u object
cols_bool = [
    (df_customers_clean, ["accepts_marketing"]),
    (df_products_clean, ["has_linked_app"]),
    (df_transactions_clean, ["is_fraud"]),
    (df_call_center_interactions_clean, ["was_resolved", "requires_followup", "was_escalated", "has_transcript", "has_recording"]),
    (df_complaints_clean, ["sla_breached", "is_repeat_complainer"])
]

for df, cols in cols_bool:
    for c in cols:
        if c in df.columns:
            df[c] = df[c].astype("boolean")

print("Tipos de datos corregidos (Fechas, Strings limpios, Enteros Nullables y Booleanos).")

# Exportacion

In [ ]:
# ==============================================================================
# EXPORTACIÓN DE DATOS LIMPIOS PARA INGESTA EN SUPABASE / POSTGRESQL
# ==============================================================================
directorio_salida = "./datos_limpios_bank_bot"
os.makedirs(directorio_salida, exist_ok=True)

# Diccionario con los dataframes a exportar
dataframes_limpios = {
    "customers_clean": df_customers_clean,
    "branches_clean": df_branches_clean,
    "products_clean": df_products_clean,
    "complaints_clean": df_complaints_clean,
    "transcripts_clean": df_transcripts_clean,
    "transactions_clean": df_transactions_clean,
    "call_center_interactions_clean": df_call_center_interactions_clean
}

print(f"{'='*50}\nINICIANDO EXPORTACIÓN A PARQUET\n{'-'*50}")

for nombre, df in dataframes_limpios.items():
    ruta_archivo = f"{directorio_salida}/{nombre}.parquet"
    # Exportamos a Parquet (requiere instalar pyarrow o fastparquet si no lo tienes: pip install pyarrow)
    df.to_parquet(ruta_archivo, index=False)
    print(f"Guardado: {ruta_archivo} ({len(df):,} filas)")

print(f"\nTodos los archivos han sido exportados exitosamente en la carpeta '{directorio_salida}'.")

### Acciones Realizadas
* Ejecución de un perfilado sobre 7 tablas representativas de un volumen de 19 millones de registros de operaciones bancarias en México, Colombia y Argentina.   
* Validación de la integridad referencial cruzando llaves primarias y foráneas entre dimensiones (clientes, sucursales, productos) y tablas de hechos (transacciones, quejas, interacciones).   
* Comprobación de hipótesis de negocio mediante tablas de contingencia (`pd.crosstab`) para aislar nulos estructurales de errores inyectados.   
* Construcción de un pipeline ETL que preserva los `DataFrames` originales en memoria y aplica reglas de limpieza sobre copias independientes (`_clean`).

### Conclusiones y Hallazgos Clave
* **Imposibilidad de NLP Predictivo Histórico:** Las 171,321 transcripciones se limitan a 42 plantillas estáticas vacías (ej. `{monto}`) sin vocabulario de fraude o reclamos. El sistema "AI-first" deberá procesar lenguaje natural exclusivamente en tiempo real desde el frontend.   
* **Umbral Determinista para la Matriz:** Toda transacción con un `fraud_score` superior a 30.0 es matemáticamente un fraude confirmado. Los valores por debajo de este umbral presentan solapamiento, dictando exactamente cuándo el agente IA debe bloquear automáticamente y cuándo debe escalar a un humano.   
* **Rupturas Lógicas Intencionales:** En el 100% de los reclamos, el cliente que interpone la queja no es el dueño del producto afectado. Además, la llave `origin_interaction_id` para rastrear quejas desde el Call Center es 100% nula.   
* **Anomalías Temporales:** Un 18.7% de las transacciones ocurrieron antes de que el cliente se registrara o de que el producto existiera.
* **Geografía y Divisas:** Las coordenadas de latitud y longitud son inválidas (oceánicas o cero). Los clientes en México operan exclusivamente en USD, y el motor del banco omitió la conversión a USD (`amount_usd`) para las operaciones en COP y ARS.   

### Estado Final de los Datos
* **Limpieza de Redundancias:** Se eliminaron las variables `latitude`, `longitude`, `merchant_category` (duplicado de `transaction_category`), `contact_reason` (duplicado de `reason_category`), `origin_interaction_id` y `registration_branch_id` (100% huérfano).
* **Imputación y Normalización:** Los valores nulos en los acentos se rellenaron con la categoría "neutral". Las variaciones de "Mexico" se estandarizaron a "México", y los 12 conflictos en `product_number` se resolvieron agregando sufijos únicos.   
* **Preservación Histórica con Banderas:** Las transacciones con desfases temporales no se eliminaron; se marcaron con las variables booleanas `flag_tx_antes_registro` y `flag_tx_antes_apertura` para que el backend suspenda las validaciones duras de fecha.
* **Listos para Ingesta:** Existen siete `DataFrames` limpios (`_clean`), libres de falsos positivos y de ruido irrecuperable, preparados para ser exportados a formato Parquet y cargados en PostgreSQL (Supabase) con la extensión `pgvector`.